# 02 — Feature Engineering

This notebook loads the cleaned Spotify datasets and creates modeling-ready features for:
- Regression: predict stream count
- Classification: predict popular vs non-popular tracks
- Clustering: discover track groups using engineered numeric features


In [3]:
# Environment and imports
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', 100)


In [4]:
# Load raw data from src (source of truth)
DATA_DIR = Path('..') / 'src'
df_album = pd.read_csv(DATA_DIR / 'spotify_skz_counts_album.csv')
df_day = pd.read_csv(DATA_DIR / 'spotify_skz_counts_day.csv')
try:
    df_stream = pd.read_csv(DATA_DIR / 'spotify_skz_streaming_4.25_4.26.csv')
except UnicodeDecodeError:
    df_stream = pd.read_csv(DATA_DIR / 'spotify_skz_streaming_4.25_4.26.csv', encoding='latin-1')

print(df_album.shape, df_day.shape, df_stream.shape)
df_album.head()


(309, 12) (1808, 11) (27005, 16)


,release_year,album_type,album_name,track_name,bpm,number_of_streams,total_time_played,total_time_played_h,average_ms_played,average_ms_played_min,song_length_min,avg_percent_song_played
0,2018,Mini Album,I am NOT,3rd Eye,140,32,7554514,2:05:55,236078.56,3:56,4:03,97
1,2018,Mini Album,I am NOT,Awaken,164,33,6399256,1:46:39,193916.85,3:13,3:13,100
2,2018,Mini Album,I am NOT,District 9,90,151,32233152,8:57:13,213464.58,3:33,3:33,100
3,2018,Mini Album,I am NOT,Grow Up,174,23,4881245,1:21:21,212228.04,3:32,3:33,100
4,2018,Mini Album,I am NOT,Mirror,186,26,5329113,1:28:49,204965.88,3:25,3:42,92


In [5]:
# Utility helpers

def parse_time_to_seconds(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    parts = str(value).split(':')
    try:
        parts = [int(p) for p in parts]
    except ValueError:
        return np.nan
    if len(parts) == 3:
        hours, minutes, seconds = parts
        return hours * 3600 + minutes * 60 + seconds
    if len(parts) == 2:
        minutes, seconds = parts
        return minutes * 60 + seconds
    return float(parts[0])


In [6]:
# Track-level engineered features
# Ensure numeric columns are numeric
for col in ['release_year', 'bpm', 'number_of_streams', 'average_ms_played', 'avg_percent_song_played', 'song_length_sec']:
    if col in df_album.columns:
        df_album[col] = pd.to_numeric(df_album[col], errors='coerce')

# Derived features
if 'release_year' in df_album.columns:
    df_album['song_age_years'] = 2026 - df_album['release_year']

if 'song_length_sec' in df_album.columns:
    df_album['song_length_min_num'] = df_album['song_length_sec'] / 60.0

if 'average_ms_played' in df_album.columns:
    df_album['average_played_sec'] = df_album['average_ms_played'] / 1000.0

if set(['bpm', 'avg_percent_song_played']).issubset(df_album.columns):
    df_album['bpm_x_completion'] = df_album['bpm'] * df_album['avg_percent_song_played']

if set(['bpm', 'song_length_sec']).issubset(df_album.columns):
    df_album['bpm_x_length'] = df_album['bpm'] * df_album['song_length_sec']

# Popularity label for classification: top 25% by streams
if 'number_of_streams' in df_album.columns:
    popularity_threshold = df_album['number_of_streams'].quantile(0.75)
    df_album['is_popular'] = (df_album['number_of_streams'] >= popularity_threshold).astype(int)

# Artist and album aggregates from the stream log
if set(['track_name', 'played_sec']).issubset(df_stream.columns):
    stream_agg = df_stream.groupby('track_name').agg(
        total_played_sec=('played_sec', 'sum'),
        avg_played_sec=('played_sec', 'mean'),
        play_count=('played_sec', 'size'),
    ).reset_index()
    df_album = df_album.merge(stream_agg, on='track_name', how='left')

# Fill simple missing engineered aggregates
for col in ['total_played_sec', 'avg_played_sec', 'play_count']:
    if col in df_album.columns:
        df_album[col] = df_album[col].fillna(0)

# Preview engineered columns
engineered_cols = [c for c in ['song_age_years', 'song_length_min_num', 'average_played_sec', 'bpm_x_completion', 'bpm_x_length', 'total_played_sec', 'avg_played_sec', 'play_count', 'is_popular'] if c in df_album.columns]
df_album[engineered_cols].head()


,song_age_years,average_played_sec,bpm_x_completion,is_popular
0,8,236.07856,13580,0
1,8,193.91685,16400,0
2,8,213.46458,9000,1
3,8,212.22804,17400,0
4,8,204.96588,17112,0


In [7]:
# Modeling tables and preprocessing pipelines
regression_df = df_album.dropna(subset=['number_of_streams']).copy()
classification_df = df_album.dropna(subset=['is_popular']).copy()

feature_cols_numeric = [
    c for c in [
        'release_year', 'bpm', 'average_ms_played', 'avg_percent_song_played',
        'song_age_years', 'song_length_min_num', 'average_played_sec',
        'bpm_x_completion', 'bpm_x_length', 'total_played_sec', 'avg_played_sec', 'play_count'
    ]
    if c in df_album.columns
]
feature_cols_categorical = [c for c in ['album_type'] if c in df_album.columns]

X_reg = regression_df[feature_cols_numeric + feature_cols_categorical].copy()
y_reg = regression_df['number_of_streams'].copy()
X_clf = classification_df[feature_cols_numeric + feature_cols_categorical].copy()
y_clf = classification_df['is_popular'].copy()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, feature_cols_numeric),
        ('cat', categorical_transformer, feature_cols_categorical),
    ],
    remainder='drop',
)

print('Regression X shape:', X_reg.shape)
print('Classification X shape:', X_clf.shape)
print('Numeric features:', feature_cols_numeric)
print('Categorical features:', feature_cols_categorical)


Regression X shape: (309, 8)
Classification X shape: (309, 8)
Numeric features: ['release_year', 'bpm', 'average_ms_played', 'avg_percent_song_played', 'song_age_years', 'average_played_sec', 'bpm_x_completion']
Categorical features: ['album_type']


In [8]:
# Quick validation and save outputs
assert not X_reg.empty, 'Regression feature matrix is empty'
assert not X_clf.empty, 'Classification feature matrix is empty'
assert set(['number_of_streams', 'is_popular']).issubset(df_album.columns)

feature_out = DATA_DIR / 'feature_engineered_album.csv'
df_album.to_csv(feature_out, index=False)
print('Saved:', feature_out)
print('Popularity threshold:', popularity_threshold)
print(df_album[['track_name', 'number_of_streams', 'is_popular']].head())


Saved: ..\src\feature_engineered_album.csv
Popularity threshold: 79.0
   track_name  number_of_streams  is_popular
0     3rd Eye                 32           0
1      Awaken                 33           0
2  District 9                151           1
3     Grow Up                 23           0
4      Mirror                 26           0
